In [1]:
from pathlib import Path
import pandas as pd

def find_repo_root(marker="README.md"):
    """Walk up from current path to find the repo root containing `marker`."""
    current = Path().resolve()
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find repo root with marker '{marker}'")

# Find repo root using a known file at the root
repo_root = find_repo_root("README.md")

# Path to the CSV file
shock_path = repo_root / Path('Dataset/Dataset_IPshocks/shocks_20250514_121541.csv')
helios_path = repo_root / Path('Dataset/Dataset_ICMECAT/helio4cast_icmecat.csv')

# Load data
Shock = pd.read_csv(shock_path)
Wind_Shock = Shock[Shock['Spacecraft'] == 'Wind']
OMNI_Shock = Shock[Shock['Spacecraft'] == 'OMNI']

Wind_helios = pd.read_csv(helios_path)

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

def GSE_sc_plot(plot_df, plot_df2, plot_title):
    # === Prepare plot_df (Shock events) ===
    event_time_dt_1 = pd.DatetimeIndex(plot_df['Time'])
    event_time_numeric_1 = (event_time_dt_1 - pd.Timestamp("1970-01-01")) / pd.Timedelta(days=1)
    event_dates_str_1 = event_time_dt_1.strftime('%Y-%m-%d')

    df_Shock = pd.DataFrame({
        'x': plot_df['SC_X'],
        'y': plot_df['SC_Y'],
        'z': plot_df['SC_Z'],
        'Event Time Numeric': event_time_numeric_1,
        'Event Time Label': event_dates_str_1,
        'Event Time': event_time_dt_1
    })

    # === Prepare plot_df2 (ICME events) ===
    event_time_dt_2 = pd.DatetimeIndex(plot_df2['sc_time'])
    event_time_numeric_2 = (event_time_dt_2 - pd.Timestamp("1970-01-01")) / pd.Timedelta(days=1)
    event_dates_str_2 = event_time_dt_2.strftime('%Y-%m-%d')

    df_helios = pd.DataFrame({
        'x': plot_df2['mo_sc_GSE_x'],
        'y': plot_df2['mo_sc_GSE_y'],
        'z': plot_df2['mo_sc_GSE_z'],
        'Event Time Numeric': event_time_numeric_2,
        'Event Time Label': event_dates_str_2,
        'Event Time': event_time_dt_2
    })

    # === Start with an empty figure ===
    fig = go.Figure()

    # === Add Shock scatter (black) ===
    fig.add_trace(go.Scatter3d(
        x=df_Shock['x'], y=df_Shock['y'], z=df_Shock['z'],
        mode='markers',
        marker=dict(size=3, color='black', opacity=0.4),
        name='Shock Catalogue',
        text=df_Shock['Event Time Label'],
        hoverinfo='text'
    ))

    # === Add ICME scatter (green) ===
    fig.add_trace(go.Scatter3d(
        x=df_helios['x'], y=df_helios['y'], z=df_helios['z'],
        mode='markers',
        marker=dict(size=3, color='green', opacity=0.6),
        name='helio4cast Catalogue', 
        text=df_helios['Event Time Label'],
        hoverinfo='text'
    ))

    # === Earth marker (blue) ===
    fig.add_trace(go.Scatter3d(
        x=[0], y=[0], z=[0],
        mode='markers+text',
        marker=dict(size=10, color='blue', line=dict(color='black', width=2)),
        text=['Earth'],
        textposition='top center',
        name='Earth',
        hoverinfo='text'
    ))

    # === L1 marker (red) ===
    fig.add_trace(go.Scatter3d(
        x=[238], y=[0], z=[0],
        mode='markers+text',
        marker=dict(size=5, color='red', line=dict(color='black', width=1)),
        text=['L1'],
        textposition='top center',
        name='L1',
        hoverinfo='text'
    ))

    # === Layout customization ===
    all_x = pd.concat([df_Shock['x'], df_helios['x'], pd.Series([-10])])
    all_y = pd.concat([df_Shock['y'], df_helios['y'], pd.Series([-10])])
    all_z = pd.concat([df_Shock['z'], df_helios['z']])

    fig.update_layout(
        title=plot_title,
        scene=dict(
            xaxis=dict(title='X [Earth radii]', range=[np.min(all_x), np.max(all_x)]),
            yaxis=dict(title='Y [Earth radii]', range=[np.min(all_y), np.max(all_y)]),
            zaxis=dict(title='Z [Earth radii]', range=[np.min(all_z), np.max(all_z)])
        ),
        legend=dict(x=0.8, y=0.9),
        margin=dict(l=0, r=0, b=0, t=50),
        width=800,
        height=600
    )

    fig.show()


In [3]:
GSE_sc_plot(Wind_Shock, Wind_helios, 'Wind Spacecraft in GSE Coordinates')

## Only use relevant time range (2023-01-01 to 2024-05-16)

In [7]:
# import the data
Shock = pd.read_csv(repo_root / Path('Dataset/Dataset_IPshocks/shocks_GFOC.csv'))

Wind_Shock = Shock[Shock['Spacecraft'] == 'Wind']
OMNI_Shock = Shock[Shock['Spacecraft'] == 'OMNI']

Wind_helios = pd.read_csv(repo_root / Path('Dataset/Dataset_ICMECAT/helio4cast_icmecat_GFOC.csv'))

GSE_sc_plot(Wind_Shock, Wind_helios, 'Wind and OMNI Spacecraft in GSE Coordinates')

# Include OMNI Data

In [24]:
def GSE_sc_plot_all(plot_df, plot_df2, plot_df3, plot_title):
    # === Prepare plot_df (Shock events, WIND) ===
    event_time_dt_1 = pd.DatetimeIndex(plot_df['Time'])
    event_time_numeric_1 = (event_time_dt_1 - pd.Timestamp("1970-01-01")) / pd.Timedelta(days=1)
    event_dates_str_1 = event_time_dt_1.strftime('%Y-%m-%d')

    df_Shock = pd.DataFrame({
        'x': plot_df['SC_X'],
        'y': plot_df['SC_Y'],
        'z': plot_df['SC_Z'],
        'Event Time Numeric': event_time_numeric_1,
        'Event Time Label': event_dates_str_1,
        'Event Time': event_time_dt_1
    })

    # === Prepare plot_df2 (ICME events) ===
    event_time_dt_2 = pd.DatetimeIndex(plot_df2['sc_time'])
    event_time_numeric_2 = (event_time_dt_2 - pd.Timestamp("1970-01-01")) / pd.Timedelta(days=1)
    event_dates_str_2 = event_time_dt_2.strftime('%Y-%m-%d')

    df_helios = pd.DataFrame({
        'x': plot_df2['mo_sc_GSE_x'],
        'y': plot_df2['mo_sc_GSE_y'],
        'z': plot_df2['mo_sc_GSE_z'],
        'Event Time Numeric': event_time_numeric_2,
        'Event Time Label': event_dates_str_2,
        'Event Time': event_time_dt_2
    })

    # === Prepare plot_df (Shock events, OMNI) ===
    event_time_dt_3 = pd.DatetimeIndex(plot_df3['Time'])
    event_time_numeric_3 = (event_time_dt_3 - pd.Timestamp("1970-01-01")) / pd.Timedelta(days=1)
    event_dates_str_3 = event_time_dt_3.strftime('%Y-%m-%d')

    df_OMNI = pd.DataFrame({
        'x': plot_df3['SC_X'],
        'y': plot_df3['SC_Y'],
        'z': plot_df3['SC_Z'],
        'Event Time Numeric': event_time_numeric_3,
        'Event Time Label': event_dates_str_3,
        'Event Time': event_time_dt_3
    })
    
    # === Start with an empty figure ===
    fig = go.Figure()

    # === Add Shock scatter (black) ===
    fig.add_trace(go.Scatter3d(
        x=df_Shock['x'], y=df_Shock['y'], z=df_Shock['z'],
        mode='markers',
        marker=dict(size=3, color='black', opacity=0.4),
        name='IPShock: WIND',
        text=df_Shock['Event Time Label'],
        hoverinfo='text'
    ))

    # === Add OMNI Shock scatter (pink) ===
    fig.add_trace(go.Scatter3d(
        x=df_OMNI['x'], y=df_OMNI['y'], z=df_OMNI['z'],
        mode='markers',
        marker=dict(size=3, color='pink', opacity=0.4),
        name='IPShock: OMNI',
        text=df_OMNI['Event Time Label'],
        hoverinfo='text'
    ))

    # === Add ICME scatter (green) ===
    fig.add_trace(go.Scatter3d(
        x=df_helios['x'], y=df_helios['y'], z=df_helios['z'],
        mode='markers',
        marker=dict(size=3, color='green', opacity=0.6),
        name='helio4cast: WIND', 
        text=df_helios['Event Time Label'],
        hoverinfo='text'
    ))

    # === Earth sphere (radius = 1 Earth radius in data units) ===
    r_earth = 1.0
    u = np.linspace(0, 2*np.pi, 40)
    v = np.linspace(0, np.pi, 20)

    x_sphere = r_earth * np.outer(np.cos(u), np.sin(v))
    y_sphere = r_earth * np.outer(np.sin(u), np.sin(v))
    z_sphere = r_earth * np.outer(np.ones_like(u), np.cos(v))

    fig.add_trace(go.Surface(
        x=x_sphere,
        y=y_sphere,
        z=z_sphere,
        colorscale=[[0, 'blue'], [1, 'blue']],
        showscale=False,
        opacity=1.0,
        name='Earth',
        hoverinfo='skip'
    ))
    # # === Earth marker (blue) ===
    # fig.add_trace(go.Scatter3d(
    #     x=[0], y=[0], z=[0],
    #     mode='markers+text',
    #     marker=dict(size=10, color='blue', line=dict(color='black', width=2)),
    #     text=['Earth'],
    #     textposition='top center',
    #     name='Earth',
    #     hoverinfo='text'
    # ))

    # === L1 marker (red) ===
    fig.add_trace(go.Scatter3d(
        x=[238], y=[0], z=[0],
        mode='markers+text',
        marker=dict(size=5, color='red', line=dict(color='black', width=1)),
        text=['L1'],
        textposition='top center',
        name='L1',
        hoverinfo='text'
    ))

    # === Layout customization ===
    all_x = pd.concat([df_Shock['x'], df_OMNI['x'], df_helios['x'], pd.Series([-20])])
    all_y = pd.concat([df_Shock['y'], df_OMNI['y'], df_helios['y'], pd.Series([-20])])
    all_z = pd.concat([df_Shock['z'], df_OMNI['z'], df_helios['z']])

    fig.update_layout(
        title=plot_title,
        scene=dict(
            aspectmode='data',
            xaxis=dict(title='X [Earth radii]', range=[np.min(all_x), np.max(all_x)]),
            yaxis=dict(title='Y [Earth radii]', range=[np.min(all_y), np.max(all_y)]),
            zaxis=dict(title='Z [Earth radii]', range=[np.min(all_z), np.max(all_z)])
        ),
        legend=dict(x=0.8, y=0.9),
        margin=dict(l=0, r=0, b=0, t=50),
        width=800,
        height=600
    )

    fig.show()


In [25]:
GSE_sc_plot_all(Wind_Shock, Wind_helios, OMNI_Shock, 'Wind and OMNI Catalogues in GSE Coordinates')